# CIDEC investigation notebook

This notebook packages the revised main-paper version of the study as an investigation focused on **strength**, **embodied carbon**, and **carbonation depth**. It uses the previously generated result tables in `results .zip` and creates the reduced figure set for the EAAI draft.

In [1]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path.cwd()
WORK = BASE / 'CIDEC_EAAI_investigation_outputs'
WORK.mkdir(exist_ok=True)
FIGDIR = WORK / 'figures'
FIGDIR.mkdir(exist_ok=True)
ZIP_PATH = BASE / 'results .zip'
EXTRACT_DIR = WORK / 'results_extracted'
if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
RES = EXTRACT_DIR / 'outputs_cidec_v4'
FIGRES = RES / 'figures_pub'
print('Using results from:', RES)


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\DELL\\Downloads\\Documents\\CVUT-Assistant Prof\\causal ML paper\\7_CausalMix-EC\\new_analysis\\tex\\CIDEC_EAAI_investigation_package\\results .zip'

## Load the reduced result tables

In [ ]:
cv = pd.read_csv(RES / 'cv_summary.csv')
cv_main = cv[cv['target'].isin(['strength_mpa','embodied_carbon_actual','carbonation_depth_mm'])].copy()
cv_main['label'] = cv_main['target'].map({
    'strength_mpa': 'Strength',
    'embodied_carbon_actual': 'Embodied carbon',
    'carbonation_depth_mm': 'Carbonation depth',
})
cv_main['validation_label'] = cv_main['split_mode'].map({'group':'Grouped CV','plain':'Plain CV'})

oof_strength = pd.read_csv(FIGRES / 'oof_strength_mpa.csv')
oof_carbon = pd.read_csv(FIGRES / 'oof_embodied_carbon_actual.csv')
oof_carb = pd.read_csv(FIGRES / 'oof_carbonation_depth_mm.csv')

causal_summary = pd.read_csv(RES / 'causal_summary.csv')
causal_effects = pd.read_csv(FIGRES / 'causal_effects_for_figures.csv')

pareto = pd.read_csv(FIGRES / 'pareto_for_figures.csv')
pareto = pareto[pareto['constraint_penalty'] <= 1e-9].copy()
pareto['pred_carbonation_plot_mm'] = pareto['pred_carbonation_depth_mm'].clip(lower=0)

cv_main


## Figure style helpers

In [ ]:
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

c_blue = '#2563eb'
c_orange = '#f59e0b'
c_green = '#10b981'
c_red = '#ef4444'
c_purple = '#7c3aed'

def savefig(fig, name):
    fig.savefig(FIGDIR / f'{name}.png', bbox_inches='tight')
    fig.savefig(FIGDIR / f'{name}.pdf', bbox_inches='tight')
    print('saved', name)

def identity(ax, x, y):
    lo = float(np.nanmin([np.nanmin(x), np.nanmin(y)]))
    hi = float(np.nanmax([np.nanmax(x), np.nanmax(y)]))
    ax.plot([lo, hi], [lo, hi], '--', color='0.35', linewidth=1.0, zorder=0)

def metrics_box(df, ax):
    mae = np.mean(np.abs(df['y_true'] - df['y_pred']))
    mask = np.isfinite(df['y_true']) & np.isfinite(df['y_pred'])
    y_true = df.loc[mask, 'y_true']
    y_pred = df.loc[mask, 'y_pred']
    ss_res = float(np.sum((y_true - y_pred)**2))
    ss_tot = float(np.sum((y_true - y_true.mean())**2))
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    cov = np.mean((df['y_true'] >= df['y_lo']) & (df['y_true'] <= df['y_hi']))
    txt = f"$R^2$ = {r2:.3f}\nMAE = {mae:.3f}\nCoverage = {cov:.1%}"
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor='0.75'))


## Figure 1: main-endpoint performance

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.4))
plot_df = cv_main.sort_values('r2')
y = np.arange(len(plot_df))
colors = [c_blue if m == 'Grouped CV' else c_green for m in plot_df['validation_label']]
ax.barh(y, plot_df['r2'], color=colors, alpha=0.9)
for i, row in enumerate(plot_df.itertuples()):
    ax.text(row.r2 + 0.015, i, f"MAE={row.mae:.3f} ({row.validation_label})", va='center', fontsize=9)
ax.set_yticks(y)
ax.set_yticklabels(plot_df['label'])
ax.set_xlabel('$R^2$')
ax.set_title('Main-endpoint predictive performance')
ax.set_xlim(0, 1.08)
ax.grid(axis='x', alpha=0.25)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c_blue, label='Grouped CV'), Patch(facecolor=c_green, label='Plain CV')], loc='lower right')
savefig(fig, 'fig01_main_endpoint_performance')
plt.show()


## Figure 2: parity plots for the three retained endpoints

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.0))
ax = axes[0]
ax.hexbin(oof_strength['y_true'], oof_strength['y_pred'], gridsize=35, mincnt=1, cmap='Blues')
identity(ax, oof_strength['y_true'], oof_strength['y_pred'])
ax.set_xlabel('Observed strength (MPa)')
ax.set_ylabel('Predicted strength (MPa)')
ax.set_title('Strength')
metrics_box(oof_strength, ax)

ax = axes[1]
ax.scatter(oof_carbon['y_true'], oof_carbon['y_pred'], s=14, alpha=0.45, color=c_green, edgecolor='none')
identity(ax, oof_carbon['y_true'], oof_carbon['y_pred'])
ax.set_xlabel('Observed embodied carbon')
ax.set_ylabel('Predicted embodied carbon')
ax.set_title('Embodied carbon')
metrics_box(oof_carbon, ax)

ax = axes[2]
ax.hexbin(oof_carb['y_true'], oof_carb['y_pred'], gridsize=35, mincnt=1, cmap='Purples')
identity(ax, oof_carb['y_true'], oof_carb['y_pred'])
ax.set_xlabel('Observed carbonation depth (mm)')
ax.set_ylabel('Predicted carbonation depth (mm)')
ax.set_title('Carbonation depth')
metrics_box(oof_carb, ax)

for ax in axes:
    ax.grid(alpha=0.2)
savefig(fig, 'fig02_parity_triptych')
plt.show()


## Figure 3: causal investigation panels

In [ ]:
strength_effects = causal_summary[causal_summary['outcome'] == 'strength_mpa'].copy()
strength_effects['label'] = strength_effects['treatment'].map({
    'fly_ash_kg_m3':'Fly ash',
    'slag_kg_m3':'Slag',
    'silica_fume_kg_m3':'Silica fume'
})
strength_effects = strength_effects.set_index('label').loc[['Fly ash','Slag','Silica fume']].reset_index()

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.1))
ax = axes[0]
ax.bar(strength_effects['label'], strength_effects['mean'], yerr=strength_effects['std'], color=[c_orange, c_blue, c_purple], capsize=4)
ax.axhline(0, color='0.35', linestyle='--', linewidth=1)
ax.set_ylabel('Mean marginal effect on strength')
ax.set_title('Average SCM effects on strength')
ax.grid(axis='y', alpha=0.2)

ax = axes[1]
d = causal_effects[(causal_effects['treatment'] == 'fly_ash_kg_m3') & (causal_effects['outcome'] == 'strength_mpa')].dropna(subset=['fly_ash_kg_m3','estimated_marginal_effect']).copy()
ax.scatter(d['fly_ash_kg_m3'], d['estimated_marginal_effect'], s=14, alpha=0.22, color=c_blue, edgecolor='none')
if len(d):
    bins = pd.qcut(d['fly_ash_kg_m3'], q=min(10, d['fly_ash_kg_m3'].nunique()), duplicates='drop')
    b = d.groupby(bins, observed=True).agg({'fly_ash_kg_m3':'median','estimated_marginal_effect':'median'}).reset_index(drop=True)
    ax.plot(b['fly_ash_kg_m3'], b['estimated_marginal_effect'], '-o', color=c_red, linewidth=2, markersize=4)
ax.axhline(0, color='0.35', linestyle='--', linewidth=1)
ax.set_xlabel('Fly ash dosage (kg/m$^3$)')
ax.set_ylabel('Marginal effect on strength')
ax.set_title('Fly ash dose effect on strength')
ax.grid(alpha=0.2)

ax = axes[2]
d = causal_effects[(causal_effects['treatment'] == 'fly_ash_kg_m3') & (causal_effects['outcome'] == 'carbonation_depth_mm')].dropna(subset=['fly_ash_kg_m3','estimated_marginal_effect']).copy()
ax.scatter(d['fly_ash_kg_m3'], d['estimated_marginal_effect'], s=14, alpha=0.22, color=c_purple, edgecolor='none')
if len(d):
    bins = pd.qcut(d['fly_ash_kg_m3'], q=min(10, d['fly_ash_kg_m3'].nunique()), duplicates='drop')
    b = d.groupby(bins, observed=True).agg({'fly_ash_kg_m3':'median','estimated_marginal_effect':'median'}).reset_index(drop=True)
    ax.plot(b['fly_ash_kg_m3'], b['estimated_marginal_effect'], '-o', color=c_green, linewidth=2, markersize=4)
ax.axhline(0, color='0.35', linestyle='--', linewidth=1)
ax.set_xlabel('Fly ash dosage (kg/m$^3$)')
ax.set_ylabel('Marginal effect on carbonation depth')
ax.set_title('Fly ash dose effect on carbonation')
ax.grid(alpha=0.2)

savefig(fig, 'fig03_causal_investigation')
plt.show()


## Figure 4: inverse-design investigation

In [ ]:
for c, maximize in [('pred_strength_mpa', True), ('pred_embodied_carbon_actual', False), ('pred_carbonation_plot_mm', False)]:
    mn, mx = pareto[c].min(), pareto[c].max()
    if maximize:
        pareto[c + '_score'] = (pareto[c] - mn) / (mx - mn)
    else:
        pareto[c + '_score'] = 1 - (pareto[c] - mn) / (mx - mn)
pareto['balanced_score'] = pareto[['pred_strength_mpa_score', 'pred_embodied_carbon_actual_score', 'pred_carbonation_plot_mm_score']].mean(axis=1)
low = pareto.sort_values('pred_embodied_carbon_actual').iloc[0].copy()
balanced = pareto.sort_values('balanced_score', ascending=False).iloc[0].copy()
high = pareto[pareto['pred_carbonation_depth_mm'] >= 0].sort_values('pred_strength_mpa', ascending=False).iloc[0].copy()

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.6))
ax = axes[0]
sc = ax.scatter(pareto['pred_embodied_carbon_actual'], pareto['pred_strength_mpa'], c=pareto['pred_carbonation_plot_mm'], cmap='viridis', s=34, alpha=0.9)
for label, row in [('A', low), ('B', balanced), ('C', high)]:
    ax.scatter(row['pred_embodied_carbon_actual'], row['pred_strength_mpa'], s=90, facecolor='none', edgecolor='black', linewidth=1.5)
    ax.text(row['pred_embodied_carbon_actual'] + 2, row['pred_strength_mpa'] + 0.8, label, fontsize=10, weight='bold')
cb = fig.colorbar(sc, ax=ax)
cb.set_label('Carbonation depth used for plotting (mm)')
ax.set_xlabel('Predicted embodied carbon')
ax.set_ylabel('Predicted strength (MPa)')
ax.set_title('Pareto investigation: strength vs carbon')
ax.grid(alpha=0.2)

ax = axes[1]
top = pareto.sort_values('balanced_score', ascending=False).head(8).copy().reset_index(drop=True)
top['candidate'] = [f'C{i+1}' for i in range(len(top))]
cols = [('cement_kg_m3','Cement', c_blue), ('fly_ash_kg_m3','Fly ash', c_orange), ('slag_kg_m3','Slag', c_green), ('silica_fume_kg_m3','Silica fume', c_red)]
bottom = np.zeros(len(top))
for col, lab, color in cols:
    ax.bar(top['candidate'], top[col], bottom=bottom, label=lab, color=color)
    bottom += top[col].to_numpy()
ax.set_ylabel('Dosage (kg/m$^3$)')
ax.set_title('Top balanced candidates: binder composition')
ax.legend(ncol=2, fontsize=8)
ax.grid(axis='y', alpha=0.2)

savefig(fig, 'fig04_inverse_design_investigation')
plt.show()


## Export reduced summary tables

In [ ]:
cv_main.to_csv(WORK / 'main_endpoint_performance.csv', index=False)
rep_df = pd.DataFrame([
    {
        'Candidate': 'Low-carbon',
        'Cement (kg/m3)': low['cement_kg_m3'],
        'Fly ash (kg/m3)': low['fly_ash_kg_m3'],
        'Slag (kg/m3)': low['slag_kg_m3'],
        'Silica fume (kg/m3)': low['silica_fume_kg_m3'],
        'Water (kg/m3)': low['water_kg_m3'],
        'Pred. strength (MPa)': low['pred_strength_mpa'],
        'Pred. embodied carbon': low['pred_embodied_carbon_actual'],
        'Pred. carbonation (mm)': max(0, low['pred_carbonation_depth_mm']),
    },
    {
        'Candidate': 'Balanced',
        'Cement (kg/m3)': balanced['cement_kg_m3'],
        'Fly ash (kg/m3)': balanced['fly_ash_kg_m3'],
        'Slag (kg/m3)': balanced['slag_kg_m3'],
        'Silica fume (kg/m3)': balanced['silica_fume_kg_m3'],
        'Water (kg/m3)': balanced['water_kg_m3'],
        'Pred. strength (MPa)': balanced['pred_strength_mpa'],
        'Pred. embodied carbon': balanced['pred_embodied_carbon_actual'],
        'Pred. carbonation (mm)': max(0, balanced['pred_carbonation_depth_mm']),
    },
    {
        'Candidate': 'High-strength',
        'Cement (kg/m3)': high['cement_kg_m3'],
        'Fly ash (kg/m3)': high['fly_ash_kg_m3'],
        'Slag (kg/m3)': high['slag_kg_m3'],
        'Silica fume (kg/m3)': high['silica_fume_kg_m3'],
        'Water (kg/m3)': high['water_kg_m3'],
        'Pred. strength (MPa)': high['pred_strength_mpa'],
        'Pred. embodied carbon': high['pred_embodied_carbon_actual'],
        'Pred. carbonation (mm)': max(0, high['pred_carbonation_depth_mm']),
    },
])
rep_df.to_csv(WORK / 'representative_candidates.csv', index=False)
print('Saved tables to', WORK)
rep_df


In [2]:
import matplotlib.pyplot as plt
import pandas as pd

# -----------------------------
# Expected input data
# -----------------------------
# pareto_df columns:
#   pred_strength
#   pred_embodied_carbon
#   pred_carbonation
#   is_pareto   (optional: True/False)
#
# rep_df columns:
#   label
#   pred_strength
#   pred_embodied_carbon
#   pred_carbonation

# Example fallback for representative candidates
# Remove this block if you already have rep_df
rep_df = pd.DataFrame({
    "label": ["Candidate A", "Candidate B", "Candidate C"],
    "pred_strength": [67.49, 86.88, 90.45],
    "pred_embodied_carbon": [138.35, 189.43, 253.88],
    "pred_carbonation": [1.63, 0.20, 0.09],
})

# -----------------------------
# Basic checks
# -----------------------------
required_cols = ["pred_strength", "pred_embodied_carbon", "pred_carbonation"]
for col in required_cols:
    if col not in pareto_df.columns:
        raise ValueError(f"Missing required column in pareto_df: {col}")

if "is_pareto" not in pareto_df.columns:
    pareto_df["is_pareto"] = False

# -----------------------------
# Create figure
# -----------------------------
fig, ax = plt.subplots(figsize=(7.2, 5.6))

# All feasible candidates
sc_all = ax.scatter(
    pareto_df["pred_embodied_carbon"],
    pareto_df["pred_strength"],
    c=pareto_df["pred_carbonation"],
    s=28,
    alpha=0.55,
    edgecolors="none"
)

# Pareto subset
pareto_only = pareto_df[pareto_df["is_pareto"] == True]
if not pareto_only.empty:
    ax.scatter(
        pareto_only["pred_embodied_carbon"],
        pareto_only["pred_strength"],
        c=pareto_only["pred_carbonation"],
        s=52,
        linewidths=0.8,
        edgecolors="black"
    )

# Representative candidates
ax.scatter(
    rep_df["pred_embodied_carbon"],
    rep_df["pred_strength"],
    c=rep_df["pred_carbonation"],
    s=110,
    linewidths=1.2,
    edgecolors="black",
    marker="D"
)

# Annotate representative candidates
for _, row in rep_df.iterrows():
    ax.annotate(
        row["label"],
        xy=(row["pred_embodied_carbon"], row["pred_strength"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=10
    )

# Axes labels
ax.set_xlabel("Predicted embodied carbon")
ax.set_ylabel("Predicted compressive strength (MPa)")

# Colorbar
cbar = plt.colorbar(sc_all, ax=ax)
cbar.set_label("Predicted carbonation depth (mm)")

# Grid
ax.grid(True, linewidth=0.5, alpha=0.4)

# Optional title off for journal style
# ax.set_title("Pareto-style trade-off among retained endpoints")

plt.tight_layout()
plt.savefig("fig04a_pareto_front.png", dpi=600, bbox_inches="tight")
plt.savefig("fig04a_pareto_front.pdf", bbox_inches="tight")
plt.show()

NameError: name 'pareto_df' is not defined